# Local Browser Automation Agent

**Stack:** Ollama · `@playwright/mcp` · Python MCP SDK · ReAct loop

### Quick-start
1. Copy `.env.example` → `.env` and set `OLLAMA_BASE_URL`
2. Edit `config.json` to tune behaviour
3. Edit `SYSTEM_PROMPT` and `TASK` in the cells below
4. **Run All Cells**

### `config.json` reference
| Key | Type | Description |
|-----|------|-------------|
| `model` | string | Ollama model name |
| `max_steps` | int | Hard cap on ReAct iterations |
| `headless` | bool | `true` = no browser window |
| `use_vision` | string | `"false"` · `"true"` · `"auto"` (see below) |
| `flash_mode` | bool | `true` = suppress `<think>` tokens (faster) |

**`use_vision` modes**
- `"false"` — screenshots hidden from model entirely
- `"true"` — screenshot auto-captured after every action round and injected into context
- `"auto"` — `browser_take_screenshot` tool exposed; model calls it when it wants

In [1]:
# ── Dependencies ──────────────────────────────────────────────────────────────
%pip install -q mcp httpx python-dotenv nest_asyncio


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Imports ───────────────────────────────────────────────────────────────────
import asyncio, base64, json, os, shutil, textwrap

import httpx
import nest_asyncio
from dotenv import load_dotenv
from IPython.display import display, Image as IPImage, Markdown

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# Jupyter already runs an event loop; nest_asyncio lets asyncio.run() work inside it
nest_asyncio.apply()
load_dotenv()

print("✓ Imports OK")

✓ Imports OK


In [3]:
# ── Config & Environment ──────────────────────────────────────────────────────
with open("config.json") as _fh:
    _cfg = json.load(_fh)

MODEL      = _cfg.get("model",      "qwen3-vl:30b")
MAX_STEPS  = int(_cfg.get("max_steps",   20))
HEADLESS   = bool(_cfg.get("headless",   False))
USE_VISION = str(_cfg.get("use_vision",  "auto"))   # "false" | "true" | "auto"
FLASH_MODE = bool(_cfg.get("flash_mode", True))     # True  → think=False in Ollama

assert USE_VISION in {"false", "true", "auto"}, (
    f"use_vision must be 'false', 'true', or 'auto' — got {USE_VISION!r}"
)

OLLAMA_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")

# ── Resolve npx path ──────────────────────────────────────────────────────────
# Jupyter subprocesses often have a stripped PATH that excludes Node's bin dir.
# Search shutil.which first, then fall back to common install locations.
_NPX_SEARCH_PATHS = [
    # Homebrew (Apple Silicon / Intel)
    "/opt/homebrew/bin/npx",
    "/usr/local/bin/npx",
    # nvm default
    os.path.expanduser("~/.nvm/versions/node/$(node --version 2>/dev/null)/bin/npx"),
    # volta
    os.path.expanduser("~/.volta/bin/npx"),
    # n / system
    "/usr/bin/npx",
]

_npx = shutil.which("npx")
if _npx is None:
    for _p in _NPX_SEARCH_PATHS:
        if os.path.isfile(_p) and os.access(_p, os.X_OK):
            _npx = _p
            break

if _npx is None:
    raise EnvironmentError(
        "npx not found. Please install Node.js:\n"
        "  • macOS:   brew install node\n"
        "  • or visit https://nodejs.org/en/download\n"
        "Then restart the Jupyter kernel."
    )

print(f"  npx        : {_npx}")

# Build @playwright/mcp server command
_mcp_args = ["@playwright/mcp@latest"]
if HEADLESS:
    _mcp_args.append("--headless")

# Pass the full environment so the subprocess inherits PATH, HOME, DISPLAY, etc.
SERVER_PARAMS = StdioServerParameters(
    command=_npx,
    args=_mcp_args,
    env=dict(os.environ),   # explicit env avoids PATH stripping in some Jupyter setups
)

print(f"  model      : {MODEL}")
print(f"  max_steps  : {MAX_STEPS}")
print(f"  headless   : {HEADLESS}")
print(f"  use_vision : {USE_VISION}")
print(f"  flash_mode : {FLASH_MODE}  →  think={not FLASH_MODE}")
print(f"  ollama url : {OLLAMA_URL}")

  npx        : /opt/homebrew/bin/npx
  model      : qwen3-vl:30b
  max_steps  : 20
  headless   : False
  use_vision : auto
  flash_mode : True  →  think=False
  ollama url : http://192.168.40.12:11434


## System Prompt

Edit the cell below to define the agent's persona, constraints, and strategy.

In [4]:
SYSTEM_PROMPT = """\
You are a precise and efficient browser automation agent.
Your goal is to complete the task the user gives you using the browser tools available to you.

Guidelines:
- Think step-by-step before each action.
- After each observation, assess your progress and decide the next action.
- Prefer the most direct path to completing the task; avoid unnecessary clicks or navigation.
- When you have fully completed the task, respond with a clear summary — do NOT call any more tools.
- If you encounter an unrecoverable error, explain it clearly and stop.
"""

## Task

Set what you want the agent to do, then run the **Run Agent** cell at the bottom.

In [5]:
TASK = "Go to https://example.com and summarise what the page says."

In [6]:
# ── Ollama API ────────────────────────────────────────────────────────────────

async def ollama_chat(
    messages: list[dict],
    tools: list[dict] | None = None,
) -> dict:
    """POST to Ollama /api/chat and return the full response dict."""
    body: dict = {
        "model":    MODEL,
        "messages": messages,
        "stream":   False,
        # think=False suppresses <think>...</think> blocks (flash_mode);
        # think=True enables them. Ollama silently ignores this on models
        # that don't support thinking tokens.
        "think":    not FLASH_MODE,
    }
    if tools:
        body["tools"] = tools

    async with httpx.AsyncClient(timeout=300) as client:
        resp = await client.post(f"{OLLAMA_URL}/api/chat", json=body)
        resp.raise_for_status()
        return resp.json()


async def check_ollama() -> None:
    """Verify Ollama is reachable and the configured model is pulled."""
    async with httpx.AsyncClient(timeout=10) as client:
        try:
            r = await client.get(f"{OLLAMA_URL}/api/tags")
            r.raise_for_status()
            available = [m["name"] for m in r.json().get("models", [])]
            if any(MODEL in m for m in available):
                print(f"✓ Ollama OK — '{MODEL}' found")
            else:
                print(f"⚠  Ollama reachable but '{MODEL}' not found locally.")
                print(f"   Run:  ollama pull {MODEL}")
                print(f"   Available: {available}")
        except Exception as exc:
            print(f"✗ Ollama unreachable at {OLLAMA_URL}: {exc}")

In [7]:
# ── Tool helpers ──────────────────────────────────────────────────────────────

# Tools that produce screenshots — visibility controlled by use_vision
_SCREENSHOT_TOOLS = {"browser_take_screenshot"}


def filter_tools(tools: list, vision_mode: str) -> list:
    """
    Control which MCP tools are exposed to the model.

    "false"  → remove screenshot tools entirely (model never sees them)
    "true"   → remove screenshot tools (loop injects screenshots automatically;
                exposing the tool would cause double-captures and loop confusion)
    "auto"   → keep all tools (model calls browser_take_screenshot on its own)
    """
    if vision_mode in ("false", "true"):
        return [t for t in tools if t.name not in _SCREENSHOT_TOOLS]
    return list(tools)


def to_ollama_tools(mcp_tools: list) -> list[dict]:
    """Convert MCP Tool objects to Ollama/OpenAI function-calling schema."""
    return [
        {
            "type": "function",
            "function": {
                "name":        t.name,
                "description": t.description or "",
                # inputSchema is already a JSON Schema dict — pass through directly
                "parameters":  t.inputSchema,
            },
        }
        for t in mcp_tools
    ]

In [8]:
# ── Observation & screenshot helpers ─────────────────────────────────────────

def extract_observation(result) -> tuple[str, list[str]]:
    """
    Flatten a CallToolResult into (text, [base64_images]).

    MCP ImageContent.data is raw base64 (no 'data:image/...' prefix).
    Ollama's 'images' field also expects raw base64 — they are wire-compatible.
    Images are displayed inline in the notebook as they are encountered.
    """
    text_parts: list[str] = []
    images: list[str] = []

    for item in result.content:
        if item.type == "text":
            text_parts.append(item.text)
        elif item.type == "image":
            images.append(item.data)           # raw base64
            display(IPImage(data=base64.b64decode(item.data)))
        elif item.type == "resource":
            res = item.resource
            text_parts.append(getattr(res, "text", str(getattr(res, "uri", res))))

    text = "\n".join(text_parts).strip() or "(empty)"
    return text, images


async def capture_screenshot(session: ClientSession) -> str | None:
    """
    Directly call browser_take_screenshot on the MCP session.
    Returns raw base64 PNG, or None on failure.
    Used internally by the loop when use_vision == 'true'.
    """
    try:
        result = await session.call_tool("browser_take_screenshot", {})
        for item in result.content:
            if item.type == "image":
                return item.data
    except Exception as exc:
        print(f"  [screenshot failed] {exc}")
    return None

In [9]:
# ── ReAct agent loop ──────────────────────────────────────────────────────────

async def run_agent(task: str, session: ClientSession) -> str:
    """
    ReAct loop:  Thought (LLM) → Action (tool call) → Observation (result) → repeat.

    Terminates when the model produces a response with no tool calls (task done),
    or when MAX_STEPS is exhausted.
    """

    # ── Discover and filter tools ─────────────────────────────────────────────
    tools_resp    = await session.list_tools()
    visible_tools = filter_tools(tools_resp.tools, USE_VISION)
    ollama_tools  = to_ollama_tools(visible_tools)

    print(f"[init] {len(visible_tools)} tools exposed to model:")
    for t in visible_tools:
        print(f"       • {t.name}")

    # ── Seed message history ──────────────────────────────────────────────────
    messages: list[dict] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": task},
    ]

    final_answer = ""

    for step in range(1, MAX_STEPS + 1):
        bar = "═" * 64
        print(f"\n{bar}")
        print(f"  Step {step} / {MAX_STEPS}")
        print(bar)

        # ── LLM call ─────────────────────────────────────────────────────────
        llm_resp      = await ollama_chat(messages, tools=ollama_tools)
        assistant_msg = llm_resp["message"]
        thought       = (assistant_msg.get("content") or "").strip()
        tool_calls    = assistant_msg.get("tool_calls") or []

        if thought:
            display(Markdown(f"**Thought:** {thought}"))

        # ── Terminal condition: no tool calls means task is done ──────────────
        if not tool_calls:
            final_answer = thought
            display(Markdown(f"---\n**Task complete:**\n\n{thought}"))
            break

        # Append assistant turn before executing its actions
        messages.append(assistant_msg)

        # ── Execute tool calls ────────────────────────────────────────────────
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            fn_args = tc["function"].get("arguments", {})
            # Guard: some models return arguments as a JSON string instead of a dict
            if isinstance(fn_args, str):
                fn_args = json.loads(fn_args)

            print(f"\n  ▶  {fn_name}")
            if fn_args:
                print(textwrap.indent(json.dumps(fn_args, indent=2), "     "))

            try:
                tool_result          = await session.call_tool(fn_name, fn_args)
                obs_text, obs_images = extract_observation(tool_result)
            except Exception as exc:
                obs_text, obs_images = f"ERROR: {exc}", []
                print(f"  [error] {obs_text}")

            preview = obs_text[:400] + ("…" if len(obs_text) > 400 else "")
            print(f"\n  ◀  {preview}")

            # Tool result message — Ollama tool messages don't carry images
            tool_msg: dict = {"role": "tool", "content": obs_text}
            if tc.get("id"):
                tool_msg["tool_call_id"] = tc["id"]
            messages.append(tool_msg)

            # If the tool returned images (e.g. auto-screenshot call in vision=auto),
            # attach them as a follow-up user message (only role that accepts images)
            if obs_images and USE_VISION != "false":
                messages.append({
                    "role":    "user",
                    "content": f"[Visual output from {fn_name}]",
                    "images":  [obs_images[0]],
                })

        # ── vision="true": auto-capture screenshot after every action round ───
        if USE_VISION == "true":
            b64 = await capture_screenshot(session)
            if b64:
                print("  [vision] auto-screenshot captured")
                display(IPImage(data=base64.b64decode(b64)))
                messages.append({
                    "role":    "user",
                    "content": "Current browser state after your actions:",
                    "images":  [b64],
                })

    else:
        # MAX_STEPS exhausted — ask for a graceful summary
        print(f"\n⚠  Max steps ({MAX_STEPS}) reached.")
        messages.append({
            "role":    "user",
            "content": "Max steps reached. Summarise what you accomplished so far.",
        })
        r            = await ollama_chat(messages)
        final_answer = (r["message"].get("content") or "").strip()
        display(Markdown(f"---\n**Summary (max steps reached):**\n\n{final_answer}"))

    return final_answer

In [10]:
# ── Run Agent ─────────────────────────────────────────────────────────────────
# This cell starts the Playwright MCP server as a subprocess, connects the MCP
# client, and runs the ReAct loop. Re-run this cell to run a new task.

async def main() -> str:
    await check_ollama()
    async with stdio_client(SERVER_PARAMS) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await run_agent(TASK, session)

result = asyncio.run(main())

✓ Ollama OK — 'qwen3-vl:30b' found
[init] 22 tools exposed to model:
       • browser_close
       • browser_resize
       • browser_console_messages
       • browser_handle_dialog
       • browser_evaluate
       • browser_file_upload
       • browser_fill_form
       • browser_install
       • browser_press_key
       • browser_type
       • browser_navigate
       • browser_navigate_back
       • browser_network_requests
       • browser_run_code
       • browser_take_screenshot
       • browser_snapshot
       • browser_click
       • browser_drag
       • browser_hover
       • browser_select_option
       • browser_tabs
       • browser_wait_for

════════════════════════════════════════════════════════════════
  Step 1 / 20
════════════════════════════════════════════════════════════════

  ▶  browser_navigate
     {
       "url": "https://example.com"
     }

  ◀  ### Ran Playwright code
```js
await page.goto('https://example.com');
```
### Page
- Page URL: https://example.com/


**Thought:** The page at https://example.com is a placeholder domain used for illustrative examples in documentation. It states:  
*"This domain is for use in illustrative examples in documents. You may use this domain in literature without prior coordination or asking for permission."*  
Additional details can be found via the "Learn more" link (https://iana.org/domains/example).  

This domain is not active and exists solely for educational purposes.

---
**Task complete:**

The page at https://example.com is a placeholder domain used for illustrative examples in documentation. It states:  
*"This domain is for use in illustrative examples in documents. You may use this domain in literature without prior coordination or asking for permission."*  
Additional details can be found via the "Learn more" link (https://iana.org/domains/example).  

This domain is not active and exists solely for educational purposes.